# Attention-Gated Fusion for Modality-Robust Affective Computing
CMU-MOSI · attention-gated fusion conditioned on a missingness mask, vs. dropout-only, static, and imputation baselines.

Run cells top to bottom. Only manual step: paste a Google Drive file id in the data-download cell (see README).

## Phase 0 — Setup

In [ ]:
import os, random, numpy as np, torch

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| Seed:", SEED)

In [ ]:
!pip install -q torch torchvision gdown scikit-learn pandas scipy thop

In [ ]:
import gdown, os

os.makedirs("data", exist_ok=True)
# Get this id from: https://drive.google.com/drive/folders/1E5kojBirtd5VbfHsFp6FYWkQunk73Nsv
# (MOSI subfolder -> aligned_50.pkl -> Get link -> "Anyone with the link")
FILE_ID = "PASTE_FILE_ID_HERE"
out_path = "data/aligned_50.pkl"

if not os.path.exists(out_path):
    try:
        gdown.download(id=FILE_ID, output=out_path, quiet=False)
    except Exception as e:
        print("Direct id download failed:", e)
        print("Falling back to URL form with fuzzy matching...")
        url = f"https://drive.google.com/file/d/{FILE_ID}/view"
        gdown.download(url=url, output=out_path, quiet=False, fuzzy=True)

assert os.path.exists(out_path) and os.path.getsize(out_path) > 0, "Download did not complete"
print("Downloaded:", os.path.getsize(out_path) / 1e6, "MB")

# Fallback if the shared file is quota-blocked: mount your own Drive with a personal copy
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/aligned_50.pkl" data/aligned_50.pkl

In [ ]:
import pickle
import numpy as np

with open("data/aligned_50.pkl", "rb") as f:
    mosi = pickle.load(f)

for split in ["train", "valid", "test"]:
    d = mosi[split]
    print(split,
          "text:", np.array(d["text"]).shape,
          "audio:", np.array(d["audio"]).shape,
          "vision:", np.array(d["vision"]).shape,
          "labels:", np.array(d["regression_labels"]).shape)

In [ ]:
import torch

MISSING_RATES = [0.0, 0.25, 0.5, 0.75]

def apply_missingness(batch, rate, modalities=("text", "audio", "vision"), generator=None):
    """
    batch: dict of {modality: tensor [B, T, D]}
    Zeros out entire modalities per-sample at `rate`, returns (masked_batch, mask)
    mask: [B, n_modalities] binary, 1 = present
    Used identically for train-time dropout augmentation and test-time evaluation.
    """
    B = next(iter(batch.values())).shape[0]
    n_mod = len(modalities)
    g = generator or torch.Generator().manual_seed(SEED)
    mask = (torch.rand(B, n_mod, generator=g) > rate).float()
    all_missing = mask.sum(dim=1) == 0
    if all_missing.any():
        keep_idx = torch.randint(0, n_mod, (all_missing.sum(),), generator=g)
        mask[all_missing, keep_idx] = 1.0
    out = {}
    for i, m in enumerate(modalities):
        out[m] = batch[m] * mask[:, i].view(-1, 1, 1).to(batch[m].device)
    return out, mask.to(device)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MOSIDataset(Dataset):
    def __init__(self, split_dict):
        self.text = torch.tensor(split_dict["text"], dtype=torch.float32)
        self.audio = torch.tensor(np.nan_to_num(split_dict["audio"]), dtype=torch.float32)
        self.vision = torch.tensor(np.nan_to_num(split_dict["vision"]), dtype=torch.float32)
        self.labels = torch.tensor(split_dict["regression_labels"], dtype=torch.float32)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return {"text": self.text[idx], "audio": self.audio[idx], "vision": self.vision[idx]}, self.labels[idx]

train_ds = MOSIDataset(mosi["train"])
valid_ds = MOSIDataset(mosi["valid"])
test_ds  = MOSIDataset(mosi["test"])

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

T_DIM, A_DIM, V_DIM = train_ds.text.shape[-1], train_ds.audio.shape[-1], train_ds.vision.shape[-1]
print("Dims — text:", T_DIM, "audio:", A_DIM, "vision:", V_DIM)

## Phase 2 — Models

In [ ]:
import torch.nn as nn

HIDDEN = 128

class ModalityEncoder(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN):
        super().__init__()
        self.rnn = nn.GRU(in_dim, hidden, batch_first=True, bidirectional=False)

    def forward(self, x):
        _, h = self.rnn(x)
        return h.squeeze(0)  # [B, hidden]

In [ ]:
class EarlyFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_t = ModalityEncoder(T_DIM); self.enc_a = ModalityEncoder(A_DIM); self.enc_v = ModalityEncoder(V_DIM)
        self.head = nn.Sequential(nn.Linear(HIDDEN*3, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, 1))

    def forward(self, batch, mask=None):
        h = torch.cat([self.enc_t(batch["text"]), self.enc_a(batch["audio"]), self.enc_v(batch["vision"])], dim=-1)
        return self.head(h).squeeze(-1)

class LateFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_t = ModalityEncoder(T_DIM); self.enc_a = ModalityEncoder(A_DIM); self.enc_v = ModalityEncoder(V_DIM)
        self.head_t = nn.Linear(HIDDEN, 1); self.head_a = nn.Linear(HIDDEN, 1); self.head_v = nn.Linear(HIDDEN, 1)

    def forward(self, batch, mask=None):
        p = torch.stack([self.head_t(self.enc_t(batch["text"])).squeeze(-1),
                          self.head_a(self.enc_a(batch["audio"])).squeeze(-1),
                          self.head_v(self.enc_v(batch["vision"])).squeeze(-1)], dim=-1)
        return p.mean(dim=-1)

class FixedWeightFusion(nn.Module):
    def __init__(self, weights=(0.4, 0.3, 0.3)):
        super().__init__()
        self.enc_t = ModalityEncoder(T_DIM); self.enc_a = ModalityEncoder(A_DIM); self.enc_v = ModalityEncoder(V_DIM)
        self.head = nn.Sequential(nn.Linear(HIDDEN*3, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, 1))
        self.w = torch.tensor(weights)

    def forward(self, batch, mask=None):
        w = self.w.to(batch["text"].device)
        h = torch.cat([self.enc_t(batch["text"])*w[0], self.enc_a(batch["audio"])*w[1], self.enc_v(batch["vision"])*w[2]], dim=-1)
        return self.head(h).squeeze(-1)

# Modality-dropout-trained fusion == EarlyFusion trained WITH apply_missingness() in the loop.
# No architecture change — isolates the dropout-training contribution.

In [ ]:
class AttentionGatedFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_t = ModalityEncoder(T_DIM); self.enc_a = ModalityEncoder(A_DIM); self.enc_v = ModalityEncoder(V_DIM)
        # gate takes [concat features ; mask] -> per-modality weights
        self.gate = nn.Sequential(
            nn.Linear(HIDDEN*3 + 3, 64), nn.ReLU(), nn.Linear(64, 3), nn.Softmax(dim=-1)
        )
        self.head = nn.Sequential(nn.Linear(HIDDEN, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, 1))

    def forward(self, batch, mask):
        ht, ha, hv = self.enc_t(batch["text"]), self.enc_a(batch["audio"]), self.enc_v(batch["vision"])
        concat = torch.cat([ht, ha, hv], dim=-1)
        gate_in = torch.cat([concat, mask], dim=-1)
        w = self.gate(gate_in)  # [B, 3], explicitly conditioned on mask, not just feature stats
        fused = w[:, 0:1]*ht + w[:, 1:2]*ha + w[:, 2:3]*hv
        return self.head(fused).squeeze(-1), w

class GatingOnlyNoDropout(AttentionGatedFusion):
    """Same arch, trained WITHOUT dropout augmentation — ablation arm."""
    pass

In [ ]:
class ImputationBaseline(nn.Module):
    """Reconstructs missing modality embeddings from available ones before fusion —
    stands in for the MMIN/TFR-Net family as the required post-2023 comparison."""
    def __init__(self):
        super().__init__()
        self.enc_t = ModalityEncoder(T_DIM); self.enc_a = ModalityEncoder(A_DIM); self.enc_v = ModalityEncoder(V_DIM)
        self.imputer = nn.Sequential(nn.Linear(HIDDEN*3, HIDDEN*3), nn.ReLU(), nn.Linear(HIDDEN*3, HIDDEN*3))
        self.head = nn.Sequential(nn.Linear(HIDDEN*3, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, 1))

    def forward(self, batch, mask):
        h = torch.cat([self.enc_t(batch["text"]), self.enc_a(batch["audio"]), self.enc_v(batch["vision"])], dim=-1)
        mask_exp = mask.repeat_interleave(HIDDEN, dim=1)
        recon = self.imputer(h)
        h_filled = h * mask_exp + recon * (1 - mask_exp)
        return self.head(h_filled).squeeze(-1)

### Training/eval loop

Note: an earlier version of this pipeline used a **fixed** training-time missingness rate (0.3) which
created a train/eval distribution mismatch — dropout-trained models never saw rates above 0.3 during
training but were evaluated up to 0.75. This version samples the training-time rate per batch from
`Uniform(0, 0.75)`, matching the evaluation range. See `manuscript.md`, Methods §2.4, for details.

In [ ]:
import random
from sklearn.metrics import accuracy_score, f1_score

def run_epoch(model, loader, optimizer, missing_rate, train_with_dropout, train=True, rate_range=(0.0, 0.75)):
    model.train() if train else model.eval()
    total_loss, preds, labels_all = 0.0, [], []
    loss_fn = nn.MSELoss()
    with torch.set_grad_enabled(train):
        for batch, y in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            y = y.to(device)
            if train and train_with_dropout:
                rate = random.uniform(*rate_range)  # sampled per batch, matches eval range
            elif not train:
                rate = missing_rate
            else:
                rate = 0.0
            batch_masked, mask = apply_missingness(batch, rate)
            out = model(batch_masked, mask)
            pred = out[0] if isinstance(out, tuple) else out
            loss = loss_fn(pred, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * y.size(0)
            preds.append(pred.detach().cpu()); labels_all.append(y.cpu())
    preds = torch.cat(preds); labels_all = torch.cat(labels_all)
    binary_acc = accuracy_score((labels_all > 0).numpy(), (preds > 0).numpy())
    f1 = f1_score((labels_all > 0).numpy(), (preds > 0).numpy())
    return total_loss / len(loader.dataset), binary_acc, f1

def train_model(model_cls, epochs=15, lr=1e-3, train_with_dropout=True, missing_rate_for_dropout=0.3, ckpt_path=None):
    model = model_cls().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_val_f1, best_state = -1, None
    for ep in range(epochs):
        train_loss, train_acc, train_f1 = run_epoch(model, train_loader, opt, missing_rate_for_dropout, train_with_dropout, train=True)
        val_loss, val_acc, val_f1 = run_epoch(model, valid_loader, opt, 0.0, False, train=False)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            if ckpt_path: torch.save(best_state, ckpt_path)
    model.load_state_dict(best_state)
    return model

In [ ]:
tiny_loader = DataLoader(torch.utils.data.Subset(train_ds, range(16)), batch_size=4)
sanity_model = AttentionGatedFusion().to(device)
sanity_opt = torch.optim.Adam(sanity_model.parameters(), lr=1e-2)
for ep in range(50):
    loss, acc, f1 = run_epoch(sanity_model, tiny_loader, sanity_opt, 0.0, False, train=True)
print(f"Final tiny-subset loss: {loss:.4f} (should approach ~0 — confirms pipeline trains)")

## Phase 3 — Results

In [ ]:
import pandas as pd
import time, glob, os

# clear any stale checkpoints from a previous (e.g. buggy fixed-rate) run before retraining
for f in glob.glob("ckpt_*.pt"):
    os.remove(f)
print("Cleared old checkpoints.")

MODELS = {
    "early_fusion": (EarlyFusion, False),
    "late_fusion": (LateFusion, False),
    "fixed_weight_fusion": (FixedWeightFusion, False),
    "dropout_only_fusion": (EarlyFusion, True),
    "gating_only_no_dropout": (GatingOnlyNoDropout, False),
    "imputation_baseline_post2023": (ImputationBaseline, True),
    "attention_gated_fusion_full": (AttentionGatedFusion, True),
}
SEEDS = [42, 123, 2024]
TOTAL_RUNS = len(MODELS) * len(SEEDS)

results = []
run_times = []
run_count = 0
grid_start = time.time()

for model_name, (model_cls, use_dropout) in MODELS.items():
    for seed in SEEDS:
        run_count += 1
        run_start = time.time()
        ckpt_path = f"ckpt_{model_name}_seed{seed}.pt"

        set_seed(seed)
        model = train_model(model_cls, train_with_dropout=use_dropout, ckpt_path=ckpt_path)

        for rate in MISSING_RATES:
            set_seed(seed)
            _, acc, f1 = run_epoch(model, test_loader, None, rate, False, train=False)
            results.append({"model": model_name, "seed": seed, "missing_rate": rate, "acc": acc, "f1": f1})

        elapsed = time.time() - run_start
        run_times.append(elapsed)
        avg_time = sum(run_times) / len(run_times)
        eta_min = (avg_time * (TOTAL_RUNS - run_count)) / 60

        print(f"[{run_count}/{TOTAL_RUNS}] {model_name} seed={seed} done in {elapsed:.1f}s "
              f"| avg/run: {avg_time:.1f}s | ETA: {eta_min:.1f} min remaining")

total_min = (time.time() - grid_start) / 60
print(f"\nAll {TOTAL_RUNS} runs complete in {total_min:.1f} minutes total.")

results_df = pd.DataFrame(results)
results_df.to_csv("results_raw.csv", index=False)
results_df.head()

In [ ]:
print(results_df.isna().sum())
print(results_df.groupby("model")["acc"].agg(["min", "max", "count"]))

In [ ]:
main_table = results_df.groupby(["model", "missing_rate"]).agg(
    acc_mean=("acc", "mean"), acc_std=("acc", "std"),
    f1_mean=("f1", "mean"), f1_std=("f1", "std")
).reset_index()
main_table["acc"] = main_table.apply(lambda r: f"{r.acc_mean:.3f} ± {r.acc_std:.3f}", axis=1)
main_table["f1"] = main_table.apply(lambda r: f"{r.f1_mean:.3f} ± {r.f1_std:.3f}", axis=1)
main_table_display = main_table.pivot(index="model", columns="missing_rate", values="acc")
main_table_display

In [ ]:
from scipy import stats

def paired_significance(df, model_a, model_b, rate, metric="f1"):
    a = df[(df.model == model_a) & (df.missing_rate == rate)][metric].values
    b = df[(df.model == model_b) & (df.missing_rate == rate)][metric].values
    t_stat, p_val = stats.ttest_rel(a, b)
    return {"model_a": model_a, "model_b": model_b, "missing_rate": rate, "t_stat": t_stat, "p_value": p_val}

sig_results = []
for rate in MISSING_RATES:
    sig_results.append(paired_significance(results_df, "attention_gated_fusion_full", "dropout_only_fusion", rate))
pd.DataFrame(sig_results)

In [ ]:
sig_gating_vs_fixed = []
for rate in MISSING_RATES:
    sig_gating_vs_fixed.append(paired_significance(results_df, "gating_only_no_dropout", "fixed_weight_fusion", rate))
pd.DataFrame(sig_gating_vs_fixed)

In [ ]:
from thop import profile
import time

def measure_efficiency(model_cls, use_dropout):
    model = model_cls().to(device).eval()
    dummy = ({"text": torch.randn(1, train_ds.text.shape[1], T_DIM).to(device),
              "audio": torch.randn(1, train_ds.audio.shape[1], A_DIM).to(device),
              "vision": torch.randn(1, train_ds.vision.shape[1], V_DIM).to(device)},
             torch.ones(1, 3).to(device))
    n_params = sum(p.numel() for p in model.parameters())
    try:
        flops, _ = profile(model, inputs=dummy, verbose=False)
    except Exception:
        flops = float("nan")  # some custom forward signatures aren't thop-traceable; note in writeup
    with torch.no_grad():
        for _ in range(10): model(*dummy)
        torch.cuda.synchronize() if device.type == "cuda" else None
        start = time.time()
        for _ in range(100): model(*dummy)
        torch.cuda.synchronize() if device.type == "cuda" else None
        latency_ms = (time.time() - start) / 100 * 1000
    return n_params, flops, latency_ms

eff_rows = []
for model_name, (model_cls, use_dropout) in MODELS.items():
    n_params, flops, latency_ms = measure_efficiency(model_cls, use_dropout)
    eff_rows.append({"model": model_name, "params": n_params, "flops": flops, "latency_ms": latency_ms})
pd.DataFrame(eff_rows)

In [ ]:
full_model = AttentionGatedFusion().to(device)
full_model.load_state_dict(torch.load("ckpt_attention_gated_fusion_full_seed42.pt"))

def eval_by_pattern(model, modalities=("text", "audio", "vision")):
    rows = []
    for missing_mod in modalities:
        model.eval()
        preds, labels_all = [], []
        with torch.no_grad():
            for batch, y in test_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                mask = torch.ones(y.size(0), 3).to(device)
                idx = modalities.index(missing_mod)
                mask[:, idx] = 0.0
                batch_masked = {m: batch[m] * mask[:, i].view(-1,1,1) for i, m in enumerate(modalities)}
                out = model(batch_masked, mask)
                pred = out[0] if isinstance(out, tuple) else out
                preds.append(pred.cpu()); labels_all.append(y)
        preds = torch.cat(preds); labels_all = torch.cat(labels_all)
        acc = accuracy_score((labels_all > 0).numpy(), (preds > 0).numpy())
        rows.append({"missing_modality": missing_mod, "acc": acc})
    return pd.DataFrame(rows)

eval_by_pattern(full_model)

In [ ]:
full_model.eval()
failures = []
with torch.no_grad():
    for batch, y in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        batch_masked, mask = apply_missingness(batch, 0.5)
        out, w = full_model(batch_masked, mask)
        wrong = ((out > 0).cpu() != (y > 0)).nonzero(as_tuple=True)[0]
        for i in wrong[:5]:
            failures.append({"true": y[i].item(), "pred": out[i].item(),
                              "gate_weights": w[i].detach().cpu().tolist(),
                              "mask": mask[i].cpu().tolist()})
        if len(failures) >= 5: break
pd.DataFrame(failures)

### Diagnostic: does the gate actually respond to the missingness mask?

Checks mean gate weight on text, split by whether text is present — this is what revealed the
gate's under-compensation pattern (Discussion §4).

In [ ]:
full_model.eval()
mask_weight_check = []
with torch.no_grad():
    for batch, y in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        for rate in [0.25, 0.5, 0.75]:
            set_seed(42)
            batch_masked, mask = apply_missingness(batch, rate)
            _, w = full_model(batch_masked, mask)
            for i in range(mask.size(0)):
                mask_weight_check.append({
                    "rate": rate, "text_present": mask[i,0].item(),
                    "gate_weight_text": w[i,0].item(), "gate_weight_audio": w[i,1].item(), "gate_weight_vision": w[i,2].item()
                })
mwdf = pd.DataFrame(mask_weight_check)
mwdf.groupby(["rate", "text_present"])[["gate_weight_text","gate_weight_audio","gate_weight_vision"]].mean()

In [ ]:
config_log = {
    "seeds": SEEDS, "missing_rates": MISSING_RATES, "batch_size": BATCH_SIZE,
    "hidden_dim": HIDDEN, "epochs": 15, "lr": 1e-3, "optimizer": "Adam",
    "hardware": str(device), "dropout_training": "uniform rate sampled per batch, range (0.0, 0.75)",
}
import json
with open("config_log.json", "w") as f: json.dump(config_log, f, indent=2)
print(json.dumps(config_log, indent=2))